# Fuel Economy EDA with sqlframe library

In [1]:
from sqlframe.base.dataframe import BaseDataFrame
from sqlframe.duckdb import DuckDBSession
from sqlframe.duckdb.types import StructType, StructField, StringType, IntegerType
from sqlframe.duckdb import Window
import sqlframe.duckdb.functions as F

# For "prettier" printing of data frame tables in jupyterlab notebook, will convert to polars data frame
import polars as pl
pl.Config.set_tbl_rows(100)
pl.Config.set_tbl_cols(50)

# sqlframe doesn't support a .toPolars() conversion method, so will "monkey patch" one here:
def _to_polars(self) -> pl.DataFrame:
    return pl.from_arrow(self.toArrow())

BaseDataFrame.toPolars = _to_polars

In [3]:
session = DuckDBSession()

In [ ]:
# data dictionary can be found at: https://www.fueleconomy.gov/feg/ws/index.shtml#vehicle
df = session.read.csv("https://www.fueleconomy.gov/feg/epadata/vehicles.csv", inferSchema=False)

In [ ]:
df.printSchema()

#### There are too many columns, so I'll limit to just the ones that I'm interested in:

In [ ]:
df_final = (
    df.select(
        F.col("year"),
        F.col("make"),
        F.col("model"),
        F.col("basemodel"),
        F.col("trany").alias("transmission_type"),
        F.col("atvType").alias("alt_fuel_adv_tech_veh_type"),
        F.col("fuelType"),
        F.col("fuelType1"),
        F.col("fuelType2"),
        F.col("charge120").alias("hours_to_fully_charge_120v"),
        F.col("charge240").alias("hours_to_fully_charge_240v"),
        F.col("city08").alias("city_mpg_fuel_type1"),
        F.col("cityA08").alias("city_mpg_fuel_type2"),
        F.col("highway08").alias("hwy_mpg_fuel_type1"),
        F.col("highwayA08").alias("hwy_mpg_fuel_type2"),
        F.col("comb08").alias("comb_mpg_fuel_type1"),
        F.col("combA08").alias("comb_mpg_fuel_type2"),
        F.col("phevCity"),
        F.col("phevHwy"),
        F.col("phevComb"),
    )
)

In [ ]:
df_final.toPolars().head()

## Let's see what distinct values are in the alternate fuel/advance technology vehicle type column:

In [ ]:
(
    df_final
    .select(
        F.col("alt_fuel_adv_tech_veh_type")
    )
    .distinct()
).show()

#### On average, how long does it take for a 2026 model electric vehicle to fully charge using 240 volt charger?

In [ ]:
(
    df_final
    .filter(
        (F.col("fuelType1").like("%Elec%")) &
        (F.col("year") == 2026)
    )
    .agg(
        F.avg(
            F.col("hours_to_fully_charge_240v")
        ).alias("Avg Hours to Fully Charge 240V")
    )
).show()

#### What is the longest time to charge a 2026 EV vehicle with a 240 volt charger?

In [ ]:
(
    df_final
    .filter(
        (F.col("fuelType1").like("%Elec%"))
        & (F.col("year") == 2026)
    )
    .agg(
        F.max(
            F.col("hours_to_fully_charge_240v")
        ).alias("Max Hours to Fully Charge 240V")
    )
).show()

#### What is the shortest time to charge a 2026 EV vehicle with a 240 volt charger?

In [ ]:
(
    df_final
    .filter(
        (F.col("fuelType1").like("%Elec%"))
        & (F.col("year") == 2026)
    )
    .agg(
        F.min(
            F.col("hours_to_fully_charge_240v")
        ).alias("Min Hours to Fully Charge 240V")
    )
).show()

#### What if I want to know the year, make, and models associated with the longest and shortest charge times?  One way is to use a `Window.partitionBy()`.

Fastest charging models:

In [ ]:
w = Window.partitionBy()  # whole dataset

(
    df_final
    .filter(
        (F.col("fuelType1").like("%Elec%"))
        & (F.col("year") == 2026)
    )
    .withColumn("min_hours", F.min("hours_to_fully_charge_240v").over(w))
    .filter(F.col("hours_to_fully_charge_240v") == F.col("min_hours"))
    .select(
        "year",
        "make",
        "model",
        "fuelType1",
        "fuelType2",
        F.col("hours_to_fully_charge_240v").alias("Min Hours to Fully Charge 240V")
    )
).show()

Slowest charging models:

In [ ]:
w = Window.partitionBy()  # whole dataset

(
    df_final
    .filter(
        (F.col("fuelType1").like("%Elec%"))
        & (F.col("year") == 2026)
    )
    .withColumn("max_hours", F.max("hours_to_fully_charge_240v").over(w))
    .filter(F.col("hours_to_fully_charge_240v") == F.col("max_hours"))
    .select(
        "year",
        "make",
        "model",
        F.col("hours_to_fully_charge_240v").alias("Max Hours to Fully Charge 240V")
    )
).show()

#### With ICE, gasoline vehicles, one would think with each passing model year, the combined MPG should be highest compared to all other previous model years.  But let's confirm by obtaining the average combined MPG for all ICE, gasoline-exclusive vehicles by model year:

In [ ]:
# +----------------------------+
# | alt_fuel_adv_tech_veh_type |
# +----------------------------+
# |            None            |
# |             EV             |
# |           Hybrid           |
# |       Plug-in Hybrid       |
# |            FFV             |
# |           Diesel           |
# |            FCV             |
# |            eFCV            |
# |            CNG             |
# |        Bifuel (CNG)        |
# |        Bifuel (LPG)        |
# +----------------------------+


(
    df_final
    .filter(
        F.col("fuelType1").like("%Gasoline%")
        & (F.col("fuelType2").isNull() | (F.col("fuelType2") == ""))
        & (
            F.col("alt_fuel_adv_tech_veh_type").isNull()
            | ~F.col("alt_fuel_adv_tech_veh_type").isin("EV", "Hybrid", "Plug-in Hybrid", "FFV", "Diesel","FCV","eFCV","CNG","Bifuel (CNG)","Bifuel (LPG)")
        )
    )
    .groupBy("year")
    .agg(F.avg("comb_mpg_fuel_type1").alias("avg_comb_mpg_fuel_type1"))
    .orderBy(F.col("year"))
).toPolars()

From above we see that recent model years' MPG aren't the highest.

Based on visually looking at the table above, we see that 2018 had highest average combined MPG.  But how do we determine the model year with highest average combined MPG using sqlframe or without having to visually look at the data?

**Using groupBy, then agg():**

In [ ]:
(
    df_final
    .filter(
        F.col("fuelType1").like("%Gasoline%")
        & (F.col("fuelType2").isNull() | (F.col("fuelType2") == ""))
        & (
            F.col("alt_fuel_adv_tech_veh_type").isNull()
            | ~F.col("alt_fuel_adv_tech_veh_type").isin("EV", "Hybrid", "Plug-in Hybrid", "FFV", "Diesel","FCV","eFCV","CNG","Bifuel (CNG)","Bifuel (LPG)")
        )
    )
    .groupBy("year")
    .agg(F.avg("comb_mpg_fuel_type1").alias("avg_comb_mpg_fuel_type1"))
    .orderBy(F.col("avg_comb_mpg_fuel_type1").desc())
    .limit(1)
).toPolars()

**Obtain average, combined MPG, but also add a new column containing the actual max average combined MPG using max(), then Window.partitionBy():**

In [ ]:
(
    df_final
    .filter(
        F.col("fuelType1").like("%Gasoline%")
        & (F.col("fuelType2").isNull() | (F.col("fuelType2") == ""))
        & (
            F.col("alt_fuel_adv_tech_veh_type").isNull()
            | ~F.col("alt_fuel_adv_tech_veh_type").isin("EV", "Hybrid", "Plug-in Hybrid", "FFV", "Diesel","FCV","eFCV","CNG","Bifuel (CNG)","Bifuel (LPG)")
        )
    )
    .groupBy("year")
    .agg(F.avg("comb_mpg_fuel_type1").alias("avg_comb_mpg_fuel_type1"))
    .withColumn(
        "max_avg_comb_mpg_fuel_type1",
        F.max("avg_comb_mpg_fuel_type1").over(Window.partitionBy())
    )
    .filter(F.col("avg_comb_mpg_fuel_type1") == F.col("max_avg_comb_mpg_fuel_type1"))
).toPolars()

## Let's obtain the top 3 vehicle makes with highest combined MPG from 2016 through 2026 model years

Create a new `rank` column which is rank within each model year.  The `within each year` is a signal or clue to us that `year` is the `partition` that we want our rank to delineate or separate on.  In most business contexts, `dense rank` is what we should be using since we want ties to have the same rank, not different rank.

In [ ]:
w = Window.partitionBy("year").orderBy(F.col("max_comb_mpg").desc())

(
    df_final
    .filter(
        F.col("fuelType1").like("%Gasoline%")
        & (
            F.col("alt_fuel_adv_tech_veh_type").isNull()
            | ~F.col("alt_fuel_adv_tech_veh_type").isin("EV", "Hybrid", "Plug-in Hybrid", "FFV", "Diesel","FCV","eFCV","CNG","Bifuel (CNG)","Bifuel (LPG)")
        )
        & (
            F.col("year").between(2016,2026)
        )
    )
    .groupBy(
        "year","make"
    )
    .agg(
        F.max(F.col("comb_mpg_fuel_type1")).alias("max_comb_mpg")
    )
    .withColumn(
        "rank", F.dense_rank().over(w)
    )
    .orderBy(
        F.col("year"),
        F.col("rank")
    )
    # Comment out .filter to see the unfiltered version
    .filter(
        F.col("rank").between(1,3)
    )
).toPolars()

In [4]:
df_cars = session.read.csv("https://jacobceles.github.io/knowledge_repo/colab_and_pyspark/cars.csv", header=True, sep=";")

In [5]:
df_cars.toPolars().tail()

Car,MPG,Cylinders,Displacement,Horsepower,Weight,Acceleration,Model,Origin
str,f64,i64,f64,f64,f64,f64,i64,str
"""Ford Mustang GL""",27.0,4,140.0,86.0,2790.0,15.6,82,"""US"""
"""Volkswagen Pickup""",44.0,4,97.0,52.0,2130.0,24.6,82,"""Europe"""
"""Dodge Rampage""",32.0,4,135.0,84.0,2295.0,11.6,82,"""US"""
"""Ford Ranger""",28.0,4,120.0,79.0,2625.0,18.6,82,"""US"""
"""Chevy S-10""",31.0,4,119.0,82.0,2720.0,19.4,82,"""US"""


## Running Total Example

In [10]:
w = (
    Window
    .partitionBy(F.col("Origin"))
    .orderBy(F.col("Model"), F.col("Car"))  # defines "running" order
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

(
    df_cars
    .withColumn("cumulative_sum_on_mpg", F.sum(F.col("MPG")).over(w))
).toPolars()

Car,MPG,Cylinders,Displacement,Horsepower,Weight,Acceleration,Model,Origin,cumulative_sum_on_mpg
str,f64,i64,f64,f64,f64,f64,i64,str,f64
"""Datsun PL510""",27.0,4,97.0,88.0,2130.0,14.5,70,"""Japan""",27.0
"""Toyota Corolla Mark ii""",24.0,4,113.0,95.0,2372.0,15.0,70,"""Japan""",51.0
"""Datsun 1200""",35.0,4,72.0,69.0,1613.0,18.0,71,"""Japan""",86.0
"""Datsun PL510""",27.0,4,97.0,88.0,2130.0,14.5,71,"""Japan""",113.0
"""Toyota Corolla""",25.0,4,113.0,95.0,2228.0,14.0,71,"""Japan""",138.0
"""Toyota Corolla 1200""",31.0,4,71.0,65.0,1773.0,19.0,71,"""Japan""",169.0
"""Datsun 510 (sw)""",28.0,4,97.0,92.0,2288.0,17.0,72,"""Japan""",197.0
"""Mazda RX2 Coupe""",19.0,3,70.0,97.0,2330.0,13.5,72,"""Japan""",216.0
"""Toyota Corolla 1600 (sw)""",27.0,4,97.0,88.0,2100.0,16.5,72,"""Japan""",243.0
